# Protein–Ligand Interaction Profiling

**BioPipelines example** — characterising how a ligand engages its pocket. After co-folding, the complex is protonated and profiled four ways: ProLIF interaction fingerprints, PLIP non-covalent contact detection, per-residue Contacts, and buried-surface SASA.

[![Documentation](https://img.shields.io/badge/docs-readthedocs-blue)](https://biopipelines.readthedocs.io/en/latest/)
[![Preprint](https://img.shields.io/badge/preprint-bioRxiv-B31B1B)](https://www.biorxiv.org/content/10.64898/2026.03.11.711024v1)

In [ ]:
# Cell 1: Install BioPipelines and micromamba
!git clone https://github.com/locbp-uzh/biopipelines
%cd biopipelines
!pip install -e ".[colab]"
!wget -q https://github.com/mamba-org/micromamba-releases/releases/latest/download/micromamba-linux-64 -O /usr/local/bin/micromamba && chmod +x /usr/local/bin/micromamba

In [ ]:
# Cell 2: Install tools
from biopipelines.pipeline import *
from biopipelines import Boltz2, Reduce, ProLIF, PLIP, Contacts, SASA

with Pipeline(project="Setup", job="InstallTools"):
    Boltz2.install()
    Reduce.install()
    ProLIF.install()
    PLIP.install()
    Contacts.install()   # ProteinEnv (PyMOL); SASA reuses it
    SASA.install()

## Cell 3: Interaction Profiling Pipeline

Co-folds the HIV-1 protease dimer with its inhibitor nelfinavir, then dissects the binding mode and **integrates the four profilers into two readouts**:

- **Reduce** adds explicit hydrogens (ProLIF/PLIP need them for H-bond detection).
- **ProLIF** computes a residue × interaction-type fingerprint.
- **PLIP** independently detects H-bonds, hydrophobic contacts, π-stacking and salt bridges (+ a PyMOL session per site).
- **Contacts** counts pocket residues within 5 Å of the ligand.
- **SASA** reports how much ligand surface the protein buries (a packing proxy).

Then two `Panda` steps turn the four outputs into a coherent picture:
1. **Per-residue interaction map** — ProLIF's fingerprint pivoted to one row per pocket residue, columns = interaction types (which residue does what).
2. **Per-complex summary** — PLIP interaction counts + Contacts contact count + SASA Δ-SASA merged into a single row.

Every profiler reads the ligand's residue **code from the Boltz2 output** (`ligand=complex`), so the code stays in sync with whatever Boltz2 wrote — no hardcoded code, and robust to the library's default code.

In [ ]:
# Cell 3: Pipeline
from biopipelines.pipeline import *
from biopipelines import (Ligand, Boltz2, Reduce, ProLIF, PLIP,
                          Contacts, SASA, Panda, Plot)

with Pipeline(project="HIVProtease", job="InteractionProfile"):
    Resources(gpu="A100", time="4:00:00", memory="16GB")
    protease = Sequence("PQITLWQRPLVTIKIGGQLKEALLDTGADDTVLEEMSLPGRWKPKMIGGIGGFIKVRQYDQILIEICGHKAIGTVLVGPTPVNIIGRNLLTQIGCTLNF",
                        ids="HIVPR")
    nelfinavir = Ligand("nelfinavir")

    # Co-fold the homodimer with the inhibitor.
    complex = Boltz2(proteins=Bundle(protease, protease), ligands=nelfinavir)

    # Reduce emits only structures (no compounds passthrough), so the profilers
    # source the ligand CODE from the Boltz2 output (`ligand=complex`).
    protonated = Reduce(structures=complex)
    fp       = ProLIF(structures=protonated, ligand=complex)
    plip     = PLIP(structures=protonated, ligand=complex)
    contacts = Contacts(structures=complex, ligand=complex, contact_threshold=5.0)
    sasa     = SASA(structures=complex, ligand=complex)

    # (1) Per-residue interaction map: ProLIF fingerprint -> one row per residue,
    #     one column per interaction type (1 = present).
    residue_map = Panda(tables=fp.tables.fingerprints,
                        operations=[Panda.pivot(index="resi",
                                                columns="interaction_type",
                                                values="present"),
                                    Panda.fillna(0)])

    # (2) Per-complex summary: PLIP counts + contact count + buried surface.
    summary = Panda(tables=[plip.tables.summary, contacts.tables.contacts, sasa.tables.sasa],
                    operations=[Panda.merge(),
                                Panda.select_columns(["id", "n_hbonds", "n_hydrophobic",
                                                      "n_pi_stacking", "n_salt_bridges",
                                                      "contacts", "delta_sasa"])])

    Plot(Plot.Bar(data=summary.tables.result,
                  x="id", y="n_hbonds",
                  title="H-bonds detected (PLIP) — HIV protease / nelfinavir",
                  xlabel="Complex", ylabel="H-bonds"))
residue_map.tables.result